In [1]:
import os
from pathlib import Path
from src.utils.file_utils import parse_file_name
from src.migration.decomposer import SqlDecomposer, DecomposerWriter
from src.migration.metadata import MetadataProcessor
from src.migration.generator import PySparkGenerator
from src.paths import *

In [2]:
from src.utils.source_rule_loader import load_all_source_rules

# input_file = PROJECT_ROOT / "docs" / "datalake_old" /"dml" / "com_r_k2_cif_alias.sql"
input_file = PROJECT_ROOT / "docs" / "datalake_old" /"dml" / "com_t_mhbos_m_client.sql"


file_name = os.path.basename(input_file).replace('.sql', '')
layer, sub_layer, source_name, base_table = parse_file_name(input_file)
output_root = PROJECT_ROOT / "output" / "migration"

source_rules = load_all_source_rules()[source_name]

In [6]:
def run_migration_pipeline():
    # ==========================================
    # BƯỚC 1: BÓC TÁCH SQL (DECOMPOSER)
    # ==========================================
    decomposer = SqlDecomposer(source_rules)
    decomposed_script = decomposer.decompose(input_file)

    # Ghi file sub-SQL ra ổ đĩa
    writer = DecomposerWriter()
    writer.write(decomposed_script, output_root / file_name)
    print(f"✅ [Bước 1] Đã bóc tách thành các block tại: {output_root / file_name / 'processing_steps'}")

    # ==========================================
    # BƯỚC 2: XỬ LÝ METADATA (PROCESSOR)
    # ==========================================
    processor = MetadataProcessor(source_rules, ai_fallback=False)

    try:
        # Hàm này sẽ phân tích AST, tự động tìm file DDL và trích xuất Schema
        pipeline_config = processor.process(decomposed_script, input_file, output_root)

        # Ghi file YAML
        metadata_output_dir = output_root / file_name / "metadata"
        processor.write_yaml(pipeline_config, metadata_output_dir)

        print(f"✅ [Bước 2] Đã xử lý Metadata thành công!")
        print(f"   -> Model nhận diện được: Model {pipeline_config['model_type']}")
        print(f"   -> Khóa (Key) nhận diện được: {pipeline_config['key']}")
        print(f"   -> File YAML đã lưu tại: {metadata_output_dir / (file_name + '.yaml')}")

    except FileNotFoundError as e:
        print(f"❌ [Lỗi Bước 2]: {e}")
        print("💡 Gợi ý: Hãy đảm bảo bạn có file DDL tương ứng tại `docs/datalake_old/ddl/com_r_k2_cif_alias.sql` hoặc cùng thư mục `dml/` để hàm trích xuất Schema hoạt động!")

if __name__ == "__main__":
    run_migration_pipeline()

✅ [Bước 1] Đã bóc tách thành các block tại: C:\Users\dungp\projects\hql_spark_bridge\output\migration\com_t_mhbos_m_client\processing_steps
✅ [Bước 2] Đã xử lý Metadata thành công!
   -> Model nhận diện được: Model 3
   -> Khóa (Key) nhận diện được: user_id
   -> File YAML đã lưu tại: C:\Users\dungp\projects\hql_spark_bridge\output\migration\com_t_mhbos_m_client\metadata\com_t_mhbos_m_client.yaml


In [10]:
print("==========================================")
print(" BƯỚC 3: SINH CODE PYSPARK (GENERATOR)")
print("==========================================")

with open(output_root / file_name / "metadata" / (file_name + '.yaml'), 'r') as f:
    pipeline_config = yaml.safe_load(f)

generator = PySparkGenerator(source_rules, output_mode="simple")
generator.generate(pipeline_config, output_root / file_name)

print("🎉 Hoàn tất toàn bộ Pipeline!")

 BƯỚC 3: SINH CODE PYSPARK (GENERATOR)
Generated DDL at C:\Users\dungp\projects\hql_spark_bridge\output\migration\com_t_mhbos_m_client\ddl\com_t_mhbos_m_client.sql
Generated DML at C:\Users\dungp\projects\hql_spark_bridge\output\migration\com_t_mhbos_m_client\dml\com_t_mhbos_m_client.py
🎉 Hoàn tất toàn bộ Pipeline!
